# 02 - Temporal Patterns: Growth, Seasonality and Drift

Three temporal structures shape everything downstream:

1. **Growth** - the marketplace roughly triples in monthly volume across the window.
2. **Seasonality** - a single enormous demand spike, and a weekly rhythm.
3. **Drift** - delivery performance changes so much over the period that a model
   trained on the past systematically mis-predicts the present.

The third is the most consequential finding in this notebook, and it directly
determines how Phase 2 must be evaluated.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from viz import save_fig, use_report_style

use_report_style()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from data_load import load_analysis
from features import build_features

In [2]:
df = build_features(load_analysis())
print(f"{len(df):,} orders  |  {df.order_purchase_timestamp.min().date()} -> "
      f"{df.order_purchase_timestamp.max().date()}")
df["month"] = df["order_purchase_timestamp"].dt.to_period("M")

99,091 orders  |  2017-01-05 -> 2018-08-30


## 2.1 Growth and the Black Friday spike

In [3]:
vol = df.groupby("month").agg(
    orders=("order_id", "size"),
    revenue=("total_price", "sum"),
    aov=("total_price", "median"),
)
print(vol.round(1).to_string())

peak = vol["orders"].idxmax()
print(f"\nbusiest month: {peak} with {vol['orders'].max():,} orders "
      f"({vol['orders'].max()/vol['orders'].median():.1f}x the median month)")

         orders    revenue   aov
month                           
2017-01     800   120312.9  88.0
2017-02    1780   247303.0  85.0
2017-03    2682   374344.3  84.9
2017-04    2404   359927.2  89.0
2017-05    3700   506071.1  87.2
2017-06    3245   433038.6  79.9
2017-07    4026   498031.5  84.5
2017-08    4331   573971.7  80.0
2017-09    4285   624401.7  85.1
2017-10    4631   664219.4  89.0
2017-11    7544  1010271.4  85.0
2017-12    5673   743914.2  89.0
2018-01    7269   950030.4  88.0
2018-02    6728   844178.7  84.9
2018-03    7211   983213.4  87.0
2018-04    6939   996647.8  90.0
2018-05    6873   996517.7  90.0
2018-06    6167   865124.3  89.9
2018-07    6292   895507.2  86.0
2018-08    6511   854686.3  80.0

busiest month: 2017-11 with 7,544 orders (1.5x the median month)


November 2017 is Black Friday. It is the single largest demand event in the
data, and - as the next section shows - the delivery system does not absorb it
cleanly.

## 2.2 Delivery performance drifts, hard

This is the finding that matters most. Mean delivery lead time is not stable:
it roughly halves across the window.

In [4]:
d = df.dropna(subset=["target_delivery_days"])
perf = d.groupby("month").agg(
    mean_days=("target_delivery_days", "mean"),
    median_days=("target_delivery_days", "median"),
    late_pct=("target_is_late", lambda s: 100 * s.mean()),
    n=("order_id", "size"),
).round(2)
print(perf.to_string())

print(f"\nworst month for lead time : {perf['mean_days'].idxmax()}  "
      f"({perf['mean_days'].max():.1f} days)")
print(f"best month for lead time  : {perf['mean_days'].idxmin()}  "
      f"({perf['mean_days'].min():.1f} days)")
print(f"worst month for lateness  : {perf['late_pct'].idxmax()}  "
      f"({perf['late_pct'].max():.1f}% late)")
print(f"best month for lateness   : {perf['late_pct'].idxmin()}  "
      f"({perf['late_pct'].min():.2f}% late)")

         mean_days  median_days  late_pct     n
month                                          
2017-01      12.65        10.73      3.07   750
2017-02      13.17        10.97      3.21  1653
2017-03      12.95        10.13      5.58  2546
2017-04      14.92        12.75      7.86  2303
2017-05      11.32         9.79      3.61  3545
2017-06      12.01        10.80      3.86  3135
2017-07      11.59        10.16      3.43  3872
2017-08      11.15         9.77      3.32  4193
2017-09      11.85        10.31      5.20  4150
2017-10      11.86        10.26      5.29  4478
2017-11      15.16        12.70     14.31  7288
2017-12      15.39        13.14      8.38  5513
2018-01      14.08        11.83      6.56  7069
2018-02      16.95        14.27     16.00  6556
2018-03      16.30        13.37     21.36  7003
2018-04      11.50         9.31      5.31  6798
2018-05      11.42         9.15      8.24  6749
2018-06       9.24         7.96      1.36  6096
2018-07       8.96         7.50      4.4

Two things stand out.

**Lead time trends down steeply.** Olist got materially better at delivery over
these twenty months. A model trained on 2017 and applied to mid-2018 will
over-predict delivery time by days, not hours.

**The late rate is volatile rather than trending.** It swings between roughly
1% and 21% month to month. Those spikes line up with Black Friday (Nov 2017)
and with the nationwide truckers' strike that paralysed Brazilian road freight
in May 2018 - shocks that no order-level feature can anticipate.

In [5]:
# Quantify the drift across the modelling split.
from features import make_splits
sp = make_splits(df)
tr_mean = sp["train"]["target_delivery_days"].mean()
te_mean = sp["test"]["target_delivery_days"].mean()
print(f"train period mean lead time : {tr_mean:.2f} days")
print(f"test  period mean lead time : {te_mean:.2f} days")
print(f"drift                       : {te_mean - tr_mean:+.2f} days")
print("\nA constant predictor fitted on the training period is therefore")
print("biased by more than five days on the test period, which is why R^2")
print("against a train-fitted baseline is negative and MAE is the honest metric.")

train period mean lead time : 13.68 days
test  period mean lead time : 8.34 days
drift                       : -5.34 days

A constant predictor fitted on the training period is therefore
biased by more than five days on the test period, which is why R^2
against a train-fitted baseline is negative and MAE is the honest metric.


In [6]:
# ---- Figure 2: volume and the delivery drift ------------------------------
fig, axes = plt.subplots(2, 1, figsize=(config.FIG_WIDTH, 5.2), sharex=True)
x = vol.index.to_timestamp()

ax = axes[0]
ax.bar(x, vol["orders"], width=25, color=config.PALETTE["primary"], alpha=0.9)
ax.set_ylabel("orders")
ax.set_title("(a) Monthly order volume, with the Black Friday 2017 spike")
ax.annotate("Black Friday", xy=(pd.Timestamp("2017-11-15"), vol["orders"].max()),
            xytext=(-6, -12), textcoords="offset points", fontsize=7.5,
            color=config.PALETTE["accent"], ha="right")

ax = axes[1]
xp = perf.index.to_timestamp()
ax.plot(xp, perf["mean_days"], color=config.PALETTE["primary"], marker="o",
        ms=3, label="mean lead time (days)")
ax.set_ylabel("days", color=config.PALETTE["primary"])
ax.tick_params(axis="y", colors=config.PALETTE["primary"])
ax2 = ax.twinx()
ax2.bar(xp, perf["late_pct"], width=20, color=config.PALETTE["accent"],
        alpha=0.35, label="% late")
ax2.set_ylabel("% delivered late", color=config.PALETTE["accent"])
ax2.tick_params(axis="y", colors=config.PALETTE["accent"])
ax2.grid(False)
ax.set_title("(b) Lead time halves over the window; lateness spikes instead")
ax.set_xlabel("month of purchase")
lines = ax.get_lines() + [ax2.patches[0]]
ax.legend(lines, ["mean lead time (days)", "% delivered late"], loc="upper right",
          fontsize=7.5)

fig.tight_layout()
print(save_fig(fig, "fig02_volume_drift"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig02_volume_drift.pdf


## 2.3 Weekly and daily rhythm

In [7]:
dow = df.groupby(df["order_purchase_timestamp"].dt.dayofweek).size()
dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
hour = df.groupby("purchase_hour").size()
print("orders by weekday:"); print(dow.to_string())
print(f"\nweekend share of orders: {100*df['is_weekend'].mean():.1f}%")
print(f"peak hour: {hour.idxmax()}:00   quietest hour: {hour.idxmin()}:00")

orders by weekday:
Mon    16141
Tue    15896
Wed    15501
Thu    14704
Fri    14074
Sat    10843
Sun    11932

weekend share of orders: 23.0%
peak hour: 16:00   quietest hour: 5:00


In [8]:
# ---- Figure 3: purchase rhythm and lead-time distribution -----------------
fig, axes = plt.subplots(1, 3, figsize=(config.FIG_WIDTH, 2.4))

axes[0].bar(dow.index, dow.values, color=config.PALETTE["primary"], alpha=0.9)
axes[0].set_title("(a) Orders by weekday", fontsize=9)
axes[0].tick_params(axis="x", labelrotation=45)

axes[1].plot(hour.index, hour.values, color=config.PALETTE["primary"])
axes[1].fill_between(hour.index, hour.values, color=config.PALETTE["primary"], alpha=0.2)
axes[1].set_title("(b) Orders by hour of day", fontsize=9)
axes[1].set_xlabel("hour")

v = d["target_delivery_days"].clip(upper=60)
axes[2].hist(v, bins=50, color=config.PALETTE["primary"], alpha=0.9)
axes[2].axvline(v.median(), color=config.PALETTE["accent"], lw=1.4)
axes[2].annotate(f"median {v.median():.0f}d", xy=(v.median(), axes[2].get_ylim()[1]*0.8),
                 xytext=(5, 0), textcoords="offset points", fontsize=7.5,
                 color=config.PALETTE["accent"])
axes[2].set_title("(c) Delivery lead time", fontsize=9)
axes[2].set_xlabel("days (clipped at 60)")

fig.tight_layout()
print(save_fig(fig, "fig03_rhythm"))

E:\2025 NUS\IT5006\IT5006 PROJECT\report\figures\fig03_rhythm.pdf


## 2.4 Takeaways

* Monthly volume roughly triples across the window; **November 2017 (Black
  Friday) is the largest single demand event**.
* **Delivery lead time halves** over twenty months. Between the training and
  test periods used later the mean falls by more than five days, so any model
  fitted on the past is biased on the present. Mean absolute error, not R^2, is
  the metric that survives this.
* **Lateness is volatile, not trending** - roughly 1% to 21% month to month,
  spiking at Black Friday and during the May 2018 truckers' strike. Those are
  exogenous shocks, and no order-level feature available at checkout can
  anticipate them. That bounds how well the late-delivery classifier can
  possibly do.

In [9]:
assert perf["mean_days"].max() > 2 * perf["mean_days"].min(), "expected strong drift"
assert perf["late_pct"].max() > 10 * perf["late_pct"].min(), "expected volatile lateness"
assert te_mean < tr_mean, "delivery is expected to improve over the window"
print("Temporal assertions passed.")

Temporal assertions passed.
